In [12]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [13]:
df = pd.read_csv(r'D:\Python\Project59\ML_Workshop_Customer360\data\customer_360_ml_workshop.csv')
display(df.head())

,CustomerID,Age,Region,TenureMonths,ContractType,InternetType,MonthlyUsageGB,MonthlyChargeEGP,NumServices,SupportCalls6M,LatePayments12M,PaperlessBilling,AutoPay,SatisfactionScore,Future12MRevenueEGP,Churn
0,CUST-00001,22,Alexandria,23,Month-to-month,DSL,317.9,580.48,5,5,0,Yes,Yes,5.0,6287.08,Yes
1,CUST-00002,59,Upper Egypt,9,Month-to-month,4G/5G,359.3,434.36,1,1,1,No,Yes,4.0,4619.91,No
2,CUST-00003,52,Giza,21,Month-to-month,DSL,176.9,391.35,3,5,1,Yes,Yes,5.0,4512.25,No
3,CUST-00004,41,Giza,12,Month-to-month,DSL,312.5,420.40,3,2,1,No,Yes,5.0,3809.65,No
4,CUST-00005,40,Delta,21,Month-to-month,4G/5G,191.6,462.73,3,4,0,No,Yes,3.0,4731.09,No


In [14]:
target = 'Future12MRevenueEGP'

X = df.drop(columns=[target, 'CustomerID'])

y = df[target]

In [15]:
X = df.drop(columns=[target, 'CustomerID', 'Churn'])

In [16]:
X = df.drop(columns=[target, 'CustomerID', 'Churn'])
y = df[target]

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (2400, 13)
X_test: (600, 13)
y_train: (2400,)
y_test: (600,)


In [18]:
def create_preprocessing_pipeline(dataframe):
    
    numerical_cols = dataframe.select_dtypes(
        include=['number']
    ).columns.tolist()
    
    categorical_cols = dataframe.select_dtypes(
        include=['object', 'category']
    ).columns.tolist()

    numerical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        ))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_cols),
            ('cat', categorical_transformer, categorical_cols)
        ],
        remainder='drop'
    )

    return preprocessor

In [19]:
preprocessor = create_preprocessing_pipeline(X)

In [20]:
models = {
    'Linear Regression': LinearRegression(),
    
    'Ridge': Ridge(alpha=1.0),
    
    'Lasso': Lasso(alpha=0.1, max_iter=10000),
    
    'KNN Regressor': KNeighborsRegressor(n_neighbors=5),
    
    'Decision Tree': DecisionTreeRegressor(
        random_state=42
    ),
    
    'Random Forest': RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    
    'Gradient Boosting': GradientBoostingRegressor(
        random_state=42
    )
}

In [21]:
results = []

trained_models = {}

for name, model in models.items():
    
    # Create a complete pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Train
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = pipeline.predict(X_test)
    
    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2
    })
    
    trained_models[name] = pipeline

c:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

In [22]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='RMSE',
    ascending=True
)

display(results_df)

,Model,MAE,RMSE,R²
6,Gradient Boosting,368.560228,464.972316,0.896807
1,Ridge,378.416604,474.914752,0.892347
0,Linear Regression,378.426990,474.928058,0.892341
2,Lasso,378.451636,474.942086,0.892335
5,Random Forest,393.554831,496.139312,0.882510
4,Decision Tree,537.060217,683.953597,0.776721
3,KNN Regressor,619.291123,783.499159,0.706998


### Part D — Revenue Regression Summary

The seven regression models were evaluated using **MAE, RMSE, and R²** to predict `Future12MRevenueEGP`.

* **Gradient Boosting** achieved the strongest overall results, with the **lowest MAE (368.56)** and **lowest RMSE (464.97)**, while also achieving the **highest R² (0.897)**.
* **Ridge, Linear Regression, and Lasso** performed very similarly, with R² values around **0.892**. This indicates that the relationship between the features and future revenue is largely captured by a linear model.
* **Random Forest** also performed well, achieving an R² of **0.883**, but it was slightly less accurate than the linear models and Gradient Boosting.
* **Decision Tree** and **KNN Regressor** had noticeably higher errors and lower R² values, indicating weaker predictive performance on this dataset.

**Overall, Gradient Boosting provided the best predictive performance among the tested models**, explaining approximately **89.7% of the variation in future 12-month revenue**.
